# O "programa dos sonhos" da FidelidadeScript

Antes de definir o analisador ou as demais fases do compilador, descrevemos o programa que queremos permitir que uma pessoa escreva. O exemplo combina uma compra, o cálculo do cashback, a concessão de pontos e operações comuns de uma conta de fidelidade.

Este programa é o contrato inicial da linguagem: as próximas etapas deverão torná-lo executável e validar as regras de negócio do domínio.

A primeira célula executável valida a ficha antes de definir o programa. Se os requisitos mínimos não forem atendidos, ela lança um erro.

In [ ]:
# ============================================================
# FICHA DE QUALIDADE — validar antes de definir o programa DSL
# ============================================================

import re

MINHA_FICHA = {
    "nome": "FidelidadeScript",
    "dominio": "programas de pontos e cashback",
    "tipos": ["inteiro", "decimal", "texto", "logico"],
    "comandos_dominio": [
        'creditar "<cliente>", <pontos>;',
        'resgatar "<cupom>", <pontos>;',
        'expirar "<cliente>", <pontos>;',
        'extrato "<cliente>";',
    ],
    "regras_de_negocio": [
        "percentual de cashback acima de 100% gera erro",
        "pontos negativos geram erro",
        "resgate acima do saldo disponível gera erro de execução",
    ],
    "programas_exemplo": [
        r'''
decimal compra = 250.0;
decimal percentual_cashback = 10.0;
decimal cashback = compra * percentual_cashback / 100.0;
inteiro pontos = 100;
se compra >= 200.0 {
    pontos = pontos + 50;
}
creditar "cliente_42", pontos;
extrato "cliente_42";
''',
        r'''
decimal percentual_cashback = 120.0;
inteiro pontos = -5;
se percentual_cashback > 100.0 {
    creditar "cliente_42", pontos;
}
''',
        r'''
inteiro pontos = 50;
creditar "cliente_42", pontos;
se pontos < 100 {
    resgatar "cupom_10", 100;
}
''',
    ],
}

def conferir_ficha(ficha):
    if not isinstance(ficha, dict):
        raise TypeError("A ficha da linguagem deve ser um dicionário.")

    def tem_texto(valor):
        return isinstance(valor, str) and bool(valor.strip())

    def quantidade_itens(campo):
        itens = ficha.get(campo)
        if not isinstance(itens, list):
            return 0
        return sum(tem_texto(item) for item in itens)

    nome = ficha.get("nome")
    dominio = ficha.get("dominio")
    identidade_valida = (
        tem_texto(nome)
        and tem_texto(dominio)
        and "todo" not in (nome + dominio).lower()
    )
    programas = ficha.get("programas_exemplo")
    tres_programas_validos = (
        isinstance(programas, list)
        and len(programas) == 3
        and all(tem_texto(programa) for programa in programas)
    )
    todos_usam_controle = (
        tres_programas_validos
        and all(
            re.search(r"(?m)^\s*(?:se|enquanto)\b", programa)
            for programa in programas
        )
    )
    checagens = [
        ("nome e domínio preenchidos", identidade_valida),
        ("pelo menos 3 tipos", quantidade_itens("tipos") >= 3),
        ("pelo menos 2 comandos do domínio", quantidade_itens("comandos_dominio") >= 2),
        ("pelo menos 2 regras de negócio", quantidade_itens("regras_de_negocio") >= 2),
        ("exatamente 3 programas-exemplo", tres_programas_validos),
        ("cada programa usa se ou enquanto", todos_usam_controle),
    ]

    for texto, ok in checagens:
        print(("✅" if ok else "⬜"), texto)

    faltam = [texto for texto, ok in checagens if not ok]
    if faltam:
        raise ValueError("Ficha da linguagem incompleta: " + "; ".join(faltam))

    print("\n🚀 Ficha completa! O programa de exemplo pode ser carregado.")


conferir_ficha(MINHA_FICHA)

# ============================================================
# PASSO 0 — O "programa dos sonhos" da FidelidadeScript
# ============================================================

PROGRAMA_FIDELIDADE = r'''
// FidelidadeScript — campanha de cashback e pontos
decimal compra = 250.0;
decimal percentual_cashback = 10.0;
decimal cashback = compra * percentual_cashback / 100.0;
inteiro pontos = 100;

se compra >= 200.0 {
    pontos = pontos + 50;
}

mostrar "Cashback calculado:";
mostrar cashback;
creditar "cliente_42", pontos;
extrato "cliente_42";
resgatar "cupom_10", 100;
expirar "cliente_42", 25;
extrato "cliente_42";
'''

print("Programa com", PROGRAMA_FIDELIDADE.count("\n") - 1, "linhas carregado em PROGRAMA_FIDELIDADE.")